In [1]:
from pathlib import Path
import time
import pandas as pd
import yfinance as yf

PROJECT_ROOT = Path.cwd()

# If Jupyter's current folder is notebooks/, use the parent folder instead.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw-data folder:", RAW_DIR)

Project root: /Users/kartiksinghai/Desktop/nse-stock-reversal-predictor
Raw-data folder: /Users/kartiksinghai/Desktop/nse-stock-reversal-predictor/data/raw


In [2]:
tickers_data = [
    ("RELIANCE.NS", "Reliance Industries", "Energy"),
    ("TCS.NS", "Tata Consultancy Services", "Information Technology"),
    ("INFY.NS", "Infosys", "Information Technology"),
    ("HDFCBANK.NS", "HDFC Bank", "Financial Services"),
    ("ICICIBANK.NS", "ICICI Bank", "Financial Services"),
    ("SBIN.NS", "State Bank of India", "Financial Services"),
    ("KOTAKBANK.NS", "Kotak Mahindra Bank", "Financial Services"),
    ("AXISBANK.NS", "Axis Bank", "Financial Services"),
    ("BAJFINANCE.NS", "Bajaj Finance", "Financial Services"),
    ("BAJAJFINSV.NS", "Bajaj Finserv", "Financial Services"),
    ("ITC.NS", "ITC", "Consumer Staples"),
    ("HINDUNILVR.NS", "Hindustan Unilever", "Consumer Staples"),
    ("BHARTIARTL.NS", "Bharti Airtel", "Telecom"),
    ("LT.NS", "Larsen & Toubro", "Industrials"),
    ("MARUTI.NS", "Maruti Suzuki", "Automobile"),
    ("M&M.NS", "Mahindra & Mahindra", "Automobile"),
    ("TATAMOTORS.NS", "Tata Motors", "Automobile"),
    ("SUNPHARMA.NS", "Sun Pharmaceutical", "Healthcare"),
    ("DRREDDY.NS", "Dr Reddy's Laboratories", "Healthcare"),
    ("CIPLA.NS", "Cipla", "Healthcare"),
    ("TITAN.NS", "Titan Company", "Consumer Discretionary"),
    ("ASIANPAINT.NS", "Asian Paints", "Consumer Discretionary"),
    ("ULTRACEMCO.NS", "UltraTech Cement", "Materials"),
    ("NTPC.NS", "NTPC", "Utilities"),
    ("POWERGRID.NS", "Power Grid Corporation", "Utilities"),
    ("ONGC.NS", "ONGC", "Energy"),
    ("COALINDIA.NS", "Coal India", "Materials"),
    ("ADANIPORTS.NS", "Adani Ports", "Industrials"),
    ("WIPRO.NS", "Wipro", "Information Technology"),
    ("TECHM.NS", "Tech Mahindra", "Information Technology"),
    ("HCLTECH.NS", "HCL Technologies", "Information Technology"),
    ("INDUSINDBK.NS", "IndusInd Bank", "Financial Services"),
    ("NESTLEIND.NS", "Nestle India", "Consumer Staples"),
    ("TATACONSUM.NS", "Tata Consumer Products", "Consumer Staples"),
    ("GRASIM.NS", "Grasim Industries", "Materials"),
    ("JSWSTEEL.NS", "JSW Steel", "Materials"),
    ("TATASTEEL.NS", "Tata Steel", "Materials"),
    ("HINDALCO.NS", "Hindalco Industries", "Materials"),
    ("APOLLOHOSP.NS", "Apollo Hospitals", "Healthcare"),
    ("EICHERMOT.NS", "Eicher Motors", "Automobile"),
    ("HEROMOTOCO.NS", "Hero MotoCorp", "Automobile"),
    ("BRITANNIA.NS", "Britannia Industries", "Consumer Staples"),
    ("DIVISLAB.NS", "Divi's Laboratories", "Healthcare"),
    ("BPCL.NS", "Bharat Petroleum", "Energy"),
]

tickers_df = pd.DataFrame(
    tickers_data,
    columns=["ticker", "company", "sector"]
)

tickers_df.to_csv(RAW_DIR / "tickers.csv", index=False)

print("Number of stocks:", len(tickers_df))
tickers_df.head()

Number of stocks: 44


,ticker,company,sector
0,RELIANCE.NS,Reliance Industries,Energy
1,TCS.NS,Tata Consultancy Services,Information Technology
2,INFY.NS,Infosys,Information Technology
3,HDFCBANK.NS,HDFC Bank,Financial Services
4,ICICIBANK.NS,ICICI Bank,Financial Services


In [3]:
START_DATE = "2015-01-01"
END_DATE = "2026-06-22"

download_log = []

for i, row in tickers_df.iterrows():
    ticker = row["ticker"]

    try:
        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            interval="1d",
            auto_adjust=False,
            progress=False,
            threads=False
        )

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] for col in df.columns]

        if df.empty:
            download_log.append({
                "ticker": ticker,
                "status": "FAILED_EMPTY",
                "rows": 0
            })
            print(f"[{i+1}/{len(tickers_df)}] {ticker}: FAILED (empty)")
            continue

        df = df.reset_index()
        df["ticker"] = ticker
        df["company"] = row["company"]
        df["sector"] = row["sector"]

        output_file = RAW_DIR / f"{ticker.replace('.NS', '')}.csv"
        df.to_csv(output_file, index=False)

        download_log.append({
            "ticker": ticker,
            "status": "OK",
            "rows": len(df)
        })

        print(f"[{i+1}/{len(tickers_df)}] {ticker}: OK — {len(df)} rows")

    except Exception as e:
        download_log.append({
            "ticker": ticker,
            "status": f"ERROR: {type(e).__name__}",
            "rows": 0
        })
        print(f"[{i+1}/{len(tickers_df)}] {ticker}: ERROR — {e}")

    # Small pause avoids hitting Yahoo too aggressively.
    time.sleep(0.5)

download_log_df = pd.DataFrame(download_log)
download_log_df.to_csv(RAW_DIR / "stock_download_log.csv", index=False)

download_log_df

[1/44] RELIANCE.NS: OK — 2832 rows
[2/44] TCS.NS: OK — 2832 rows
[3/44] INFY.NS: OK — 2832 rows
[4/44] HDFCBANK.NS: OK — 2832 rows
[5/44] ICICIBANK.NS: OK — 2832 rows
[6/44] SBIN.NS: OK — 2832 rows
[7/44] KOTAKBANK.NS: OK — 2832 rows
[8/44] AXISBANK.NS: OK — 2832 rows
[9/44] BAJFINANCE.NS: OK — 2832 rows
[10/44] BAJAJFINSV.NS: OK — 2832 rows
[11/44] ITC.NS: OK — 2832 rows
[12/44] HINDUNILVR.NS: OK — 2832 rows
[13/44] BHARTIARTL.NS: OK — 2832 rows
[14/44] LT.NS: OK — 2832 rows
[15/44] MARUTI.NS: OK — 2832 rows
[16/44] M&M.NS: OK — 2832 rows


HTTP Error 404: 

1 Failed download:
['TATAMOTORS.NS']: YFTzMissingError('possibly delisted; no timezone found')


[17/44] TATAMOTORS.NS: FAILED (empty)
[18/44] SUNPHARMA.NS: OK — 2832 rows
[19/44] DRREDDY.NS: OK — 2832 rows
[20/44] CIPLA.NS: OK — 2832 rows
[21/44] TITAN.NS: OK — 2832 rows
[22/44] ASIANPAINT.NS: OK — 2832 rows
[23/44] ULTRACEMCO.NS: OK — 2832 rows
[24/44] NTPC.NS: OK — 2832 rows
[25/44] POWERGRID.NS: OK — 2832 rows
[26/44] ONGC.NS: OK — 2832 rows
[27/44] COALINDIA.NS: OK — 2832 rows
[28/44] ADANIPORTS.NS: OK — 2832 rows
[29/44] WIPRO.NS: OK — 2832 rows
[30/44] TECHM.NS: OK — 2832 rows
[31/44] HCLTECH.NS: OK — 2832 rows
[32/44] INDUSINDBK.NS: OK — 2832 rows
[33/44] NESTLEIND.NS: OK — 2832 rows
[34/44] TATACONSUM.NS: OK — 2832 rows
[35/44] GRASIM.NS: OK — 2832 rows
[36/44] JSWSTEEL.NS: OK — 2832 rows
[37/44] TATASTEEL.NS: OK — 2832 rows
[38/44] HINDALCO.NS: OK — 2832 rows
[39/44] APOLLOHOSP.NS: OK — 2832 rows
[40/44] EICHERMOT.NS: OK — 2832 rows
[41/44] HEROMOTOCO.NS: OK — 2832 rows
[42/44] BRITANNIA.NS: OK — 2832 rows
[43/44] DIVISLAB.NS: OK — 2831 rows
[44/44] BPCL.NS: OK — 2832 ro

,ticker,status,rows
0,RELIANCE.NS,OK,2832
1,TCS.NS,OK,2832
2,INFY.NS,OK,2832
3,HDFCBANK.NS,OK,2832
4,ICICIBANK.NS,OK,2832
5,SBIN.NS,OK,2832
6,KOTAKBANK.NS,OK,2832
7,AXISBANK.NS,OK,2832
8,BAJFINANCE.NS,OK,2832
9,BAJAJFINSV.NS,OK,2832


In [4]:
import time
import yfinance as yf
import pandas as pd

ticker = "TATAMOTORS.NS"

time.sleep(5)

tata = yf.download(
    ticker,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    progress=False,
    threads=False
)

if isinstance(tata.columns, pd.MultiIndex):
    tata.columns = [col[0] for col in tata.columns]

print("Rows downloaded:", len(tata))
tata.head()

HTTP Error 404: 

1 Failed download:
['TATAMOTORS.NS']: YFTzMissingError('possibly delisted; no timezone found')


Rows downloaded: 0


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,


In [5]:
tickers_df = tickers_df[tickers_df["ticker"] != "TATAMOTORS.NS"].copy()

# Save the corrected universe
tickers_df.to_csv(RAW_DIR / "tickers.csv", index=False)

print("Stocks remaining:", len(tickers_df))

Stocks remaining: 43


In [6]:
download_log_df.loc[
    download_log_df["ticker"] == "TATAMOTORS.NS",
    "status"
] = "EXCLUDED_YAHOO_DOWNLOAD_FAILURE"

download_log_df.to_csv(RAW_DIR / "stock_download_log.csv", index=False)

download_log_df[download_log_df["ticker"] == "TATAMOTORS.NS"]

,ticker,status,rows
16,TATAMOTORS.NS,EXCLUDED_YAHOO_DOWNLOAD_FAILURE,0


In [7]:
nifty = yf.download(
    "^NSEI",
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    progress=False,
    threads=False
)

if isinstance(nifty.columns, pd.MultiIndex):
    nifty.columns = [col[0] for col in nifty.columns]

nifty = nifty.reset_index()
nifty.to_csv(RAW_DIR / "NIFTY50.csv", index=False)

print("NIFTY 50 rows:", len(nifty))
nifty.head()

NIFTY 50 rows: 2821


,Date,Adj Close,Close,High,Low,Open,Volume
0,2015-01-02,8395.450195,8395.450195,8410.599609,8288.700195,8288.700195,101900
1,2015-01-05,8378.400391,8378.400391,8445.599609,8363.900391,8407.950195,118200
2,2015-01-06,8127.350098,8127.350098,8327.849609,8111.350098,8325.299805,172800
3,2015-01-07,8102.100098,8102.100098,8151.200195,8065.450195,8118.649902,164100
4,2015-01-08,8234.599609,8234.599609,8243.500000,8167.299805,8191.399902,143800


In [8]:
summary = {
    "stocks_requested": len(tickers_df),
    "stocks_downloaded": (download_log_df["status"] == "OK").sum(),
    "stocks_failed": (download_log_df["status"] != "OK").sum(),
    "total_stock_rows": download_log_df["rows"].sum(),
    "nifty_rows": len(nifty)
}

pd.Series(summary)

stocks_requested         43
stocks_downloaded        43
stocks_failed             1
total_stock_rows     121775
nifty_rows             2821
dtype: int64